In [ ]:
import os, subprocess, sys, time
os.chdir('/kaggle/working')
print('ETAPE1: clone + deps')
if not os.path.isdir('/kaggle/working/ComfyUI'):
    for a in range(3):
        r = subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git'],capture_output=True,text=True)
        print('clone try',a,'rc=',r.returncode,(r.stderr or '')[-200:])
        if r.returncode == 0: break
        time.sleep(10)
else:
    print('ComfyUI deja present')
if os.path.isdir('/kaggle/working/ComfyUI'):
    r = subprocess.run([sys.executable,'-m','pip','install','-q','-r','ComfyUI/requirements.txt'],capture_output=True,text=True)
    print('req rc=', r.returncode, (r.stderr or '')[-200:])
    r = subprocess.run([sys.executable,'-m','pip','install','-q','aiohttp'],capture_output=True,text=True)
    print('aiohttp rc=', r.returncode)
else:
    print('CLONE ECHOUE - les cellules suivantes loggeront')
print('ETAPE1 FINIE')


In [ ]:
print('ETAPE2: assets manifest (kaggle/input d abord, reprise+atomique sinon)')
import os, subprocess, pathlib, shutil, time
MANIFEST = {"profile": "video", "description": "Kit video minimal eprouve: LTXV 2B + T5XXL (~11 Go, sous quota)", "checkpoints": {}, "loras": {}, "custom_nodes": [], "models_extra": {"ltxv_checkpoint": {"dir": "checkpoints", "files": {"ltx-video-2b-v0.9.5.safetensors": "/kaggle/input/models-ltx/ltx-video-2b-v0.9.5.safetensors"}}, "ltxv_text_encoder": {"dir": "text_encoders", "files": {"t5xxl_fp8_e4m3fn.safetensors": "/kaggle/input/models-ltx/t5xxl_fp8_e4m3fn.safetensors"}}}, "workflows": ["Comfy-Org templates: ltxv_text_to_video / ltxv_image_to_video (built-in Template Library)"]}
MIN_SIZE = 1e6
EXPECTED = {
 'ltx-video-2b-v0.9.5.safetensors': 5_800_000_000,
 't5xxl_fp8_e4m3fn.safetensors': 4_500_000_000,
}
def _ok(dst, name):
    return os.path.exists(dst) and os.path.getsize(dst) > max(MIN_SIZE, EXPECTED.get(name, 0)*0.9)
def _find(name, dst):
    for root in ('/kaggle/input', '/kaggle/working'):
        if not os.path.exists(root): continue
        for p in pathlib.Path(root).rglob(name):
            try:
                if p.is_file() and p.stat().st_size > MIN_SIZE and str(p) != dst:
                    return str(p)
            except OSError: pass
    return None
def _log(m):
    open('/kaggle/working/_bootlog.txt','a').write(str(m)+chr(10))
def _dl(d, files):
    os.makedirs(d, exist_ok=True)
    for name, url in files.items():
        dst = os.path.join(d, name)
        if _ok(dst, name):
            print('present:', name); _log('present '+name); continue
        found = _find(name, dst)
        if found:
            shutil.copy2(found, dst); print('copied:', name, '<-', found); _log('copied '+name); continue
        if url.startswith('/kaggle/input/'):
            shutil.copy(url, dst); print('dataset:', name); continue
        tmp = dst + '.part'
        ok = False
        for attempt in range(6):
            _log('wget start '+name+' try '+str(attempt))
            r = subprocess.run(['wget','-q','-c','--tries=5','--timeout=60','-O',tmp,url],capture_output=True,text=True)
            sz = os.path.getsize(tmp) if os.path.exists(tmp) else 0
            _log('wget end '+name+' rc='+str(r.returncode)+' size='+str(sz))
            if r.returncode == 0 and sz > max(MIN_SIZE, EXPECTED.get(name,0)*0.9):
                ok = True; break
            print(name, 'attempt', attempt, 'rc=', r.returncode, 'size=', sz, (r.stderr or '')[-120:])
            _log('retry '+name+' err='+(r.stderr or '')[-120:])
            time.sleep(5)
        if not ok:
            _log('FAILED '+name); print('FAILED:', name); continue
        os.replace(tmp, dst)
        print('downloaded:', name, os.path.getsize(dst)); _log('downloaded '+name+' '+str(os.path.getsize(dst)))
base='/kaggle/working/ComfyUI/models'
_dl(base+'/checkpoints', MANIFEST.get('checkpoints', {}))
_dl(base+'/loras', MANIFEST.get('loras', {}))
for grp in (MANIFEST.get('models_extra') or {}).values():
    if 'files' in grp:
        _dl(base+'/'+grp['dir'], grp['files'])
for repo in MANIFEST.get('custom_nodes', []):
    dest='/kaggle/working/ComfyUI/custom_nodes/'+repo.rstrip('/').split('/')[-1]
    if not os.path.exists(dest):
        subprocess.run(['git','clone','--depth','1',repo,dest],capture_output=True)
        req=os.path.join(dest,'requirements.txt')
        if os.path.exists(req):
            subprocess.run(['pip','install','-q','-r',req],capture_output=True)
        print('node:', repo.rsplit('/',1)[-1])
print('ckpts:', os.listdir(base+'/checkpoints'))
print('text_encoders:', os.listdir(base+'/text_encoders') if os.path.exists(base+'/text_encoders') else [])
print('ETAPE2 FINIE')


In [ ]:
print('ETAPE3: comfy + proxy token-gate')
import os, threading, subprocess, sys, time, asyncio, signal
from aiohttp import web, ClientSession
TOKEN = os.environ.get('HEPHAISTOS_TOKEN') or 'HaX8_jTy4snVvHItiVRdABwLV-2S2nuFc6neYqaV27E'
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HEPHAISTOS_TOKEN') or TOKEN
    print('token via kaggle secrets')
except Exception as e:
    print('token via env/placeholder:', type(e).__name__)
comfy = None
try:
    comfy = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-cuda-malloc'],
        cwd='/kaggle/working/ComfyUI')
except Exception as e:
    print('comfy start FAIL:', e)
async def proxy(req):
    if req.headers.get('X-Hephaistos-Token') != TOKEN:
        return web.Response(status=401, text='bad token')
    _last[0] = time.time()
    async with ClientSession() as s:
        url = 'http://127.0.0.1:8188' + req.path_qs
        body = await req.read() if req.can_read_body else None
        async with s.request(req.method, url, data=body,
            headers={k:v for k,v in req.headers.items() if k.lower() not in ('host','x-hephaistos-token')}) as up:
            return web.Response(status=up.status, body=await up.read(),
                headers={k:v for k,v in up.headers.items() if k.lower() not in ('content-encoding','content-length','transfer-encoding')})
async def shutdown(req):
    if req.headers.get('X-Hephaistos-Token') != TOKEN:
        return web.Response(status=401, text='bad token')
    threading.Timer(2.0, lambda: os.kill(os.getpid(), signal.SIGTERM)).start()
    return web.Response(text='extinction dans 2s')
async def main():
    app = web.Application()
    app.router.add_route('*','/shutdown',shutdown)
    app.router.add_route('*','/{path:.*}',proxy)
    r = web.AppRunner(app); await r.setup()
    await web.TCPSite(r,'0.0.0.0',8080).start()
    print('proxy pret :8080')
threading.Thread(target=lambda: asyncio.run(main()), daemon=True).start()
MAX_IDLE_S = float(os.environ.get('KERNEL_MAX_IDLE_SECONDS','7200'))
_last = [time.time()]
def _idle_guard():
    while True:
        time.sleep(30)
        if time.time() - _last[0] > MAX_IDLE_S:
            print('IDLE', MAX_IDLE_S, 's -> auto-extinction', flush=True); os.kill(os.getpid(), signal.SIGTERM)
threading.Thread(target=_idle_guard, daemon=True).start()
print('auto-extinction si idle >', MAX_IDLE_S, 's')
time.sleep(20)
print('comfy poll:', (comfy.poll() if comfy else 'COMFY ABSENT'), '(None = vivant)')
print('ETAPE3 FINIE')


In [ ]:
print('ETAPE4: tunnel')
import subprocess, re, time, threading, os
import json as _json, os as _os
_os.makedirs('/kaggle/working/.cloudflared', exist_ok=True)
_creds = _json.loads(r'''{"AccountTag":"b016b298f6355c39943c542aca0414b8","TunnelSecret":"1005SCvKEZHlIj+UWGfnfgmyQ1eCLPsio+nhKupmtPM=","TunnelID":"6a056836-2022-4863-b8a2-57a9c4cfbe31","Endpoint":""}''')
open('/kaggle/working/.cloudflared/tunnel.json','w').write(_json.dumps(_creds))
_cfg = chr(10).join(['tunnel: 6a056836-2022-4863-b8a2-57a9c4cfbe31',
 'credentials-file: /kaggle/working/.cloudflared/tunnel.json',
 'ingress:',
 '  - hostname: comfi.art-qalam.fr',
 '    service: http://127.0.0.1:8080',
 '  - service: http_status:404', ''])
open('/kaggle/working/cf-config.yml','w').write(_cfg)
print('cf config ecrite: comfi.art-qalam.fr -> :8188')
if not os.path.exists('/kaggle/working/cloudflared'):
    r = subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/kaggle/working/cloudflared'],capture_output=True,text=True)
    print('cloudflared dl rc=', r.returncode)
subprocess.run(['chmod','+x','/kaggle/working/cloudflared'],check=False)
_cf_cmd = ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--config','/kaggle/working/cf-config.yml','run','6a056836-2022-4863-b8a2-57a9c4cfbe31']
proc = subprocess.Popen(_cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for i, line in enumerate(proc.stdout):
    print('CF:', line.strip()[:160])
    if 'Registered tunnel connection' in line:
        print('>>> TUNNEL NOMME ACTIF: https://comfi.art-qalam.fr')
        break
    if i > 60: break


In [ ]:
# WATCHDOG + KEEP-ALIVE : relance cloudflared ET comfy s'ils meurent
import time, subprocess, os, signal, re, threading, sys
cf = [None]
def start_cf():
    _cf_cmd = ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--config','/kaggle/working/cf-config.yml','run','6a056836-2022-4863-b8a2-57a9c4cfbe31']
    cf[0] = subprocess.Popen(_cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def pump():
        for line in cf[0].stdout:
            if 'Registered tunnel connection' in line:
                print('>>> TUNNEL ACTIF: https://comfi.art-qalam.fr', flush=True)
            elif ' ERR ' in line or 'error' in line.lower():
                print('CF-ERR:', line.strip()[:140], flush=True)
    threading.Thread(target=pump, daemon=True).start()
def start_comfy():
    global comfy
    comfy = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-cuda-malloc'],
        cwd='/kaggle/working/ComfyUI')
print('watchdog actif', flush=True)
while True:
    if cf[0] is None or cf[0].poll() is not None:
        print('WATCHDOG: cloudflared mort -> redemarrage', flush=True)
        start_cf()
    if comfy is None or comfy.poll() is not None:
        print('WATCHDOG: comfy mort -> redemarrage', flush=True)
        start_comfy()
    time.sleep(20)
